In [1]:
import os
from dotenv import load_dotenv
import mysql.connector
import pandas as pd
import numpy as np
import joblib
import mysql.connector
from sklearn.preprocessing import LabelEncoder

load_dotenv()

conn = mysql.connector.connect(
    host=os.getenv('DB_HOST'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
    database=os.getenv('DB_NAME'),
    port=int(os.getenv('DB_PORT', 3306))
)

print("✅ Connected to database securely!")
print(f"Connected to: {os.getenv('DB_NAME')} at {os.getenv('DB_HOST')}")

✅ Connected to database securely!
Connected to: hospital_management at localhost


In [2]:
xgb = joblib.load('C:/Users/Tanishq Verma/OneDrive/Documents/GitHub/Hospital-Management-System-using-Queuing-theory/models/saved/xgboost_model.pkl')

le = LabelEncoder()
le.fit(['Cardiac Ward', 'Cardiology Ward', 'Emergency Ward', 
        'Endocrinology Ward', 'General Ward', 'ICU'])

print("✅ Models loaded!")

✅ Models loaded!


In [3]:
def predict_and_assign_ward(patient_features, patient_id):

    prediction_encoded = xgb.predict([patient_features])[0]
    predicted_ward = le.inverse_transform([prediction_encoded])[0]
    
    
    proba = xgb.predict_proba([patient_features])
    confidence = float(max(proba[0]))
    
    
    cursor = conn.cursor()
    
    try:
        
        cursor.execute("SELECT ward_id, available_beds FROM wards WHERE ward_name = %s", (predicted_ward,))
        ward_data = cursor.fetchone()
        
        if not ward_data:
            return {'success': False, 'message': f'Ward {predicted_ward} not found'}
        
        ward_id, available_beds = ward_data
        
        if available_beds <= 0:
            return {'success': False, 'message': f'No beds available in {predicted_ward}'}
        
        
        cursor.execute("""
            SELECT bed_id, bed_number FROM beds 
            WHERE ward_id = %s AND is_occupied = 0 
            LIMIT 1
        """, (ward_id,))
        
        bed_data = cursor.fetchone()
        if not bed_data:
            return {'success': False, 'message': 'No beds available'}
        
        bed_id, bed_number = bed_data
        
        
        patient_id = str(patient_id)
        ward_id = int(ward_id)
        bed_id = int(bed_id)
        
        
        cursor.execute("""
            INSERT INTO patient_assignments 
            (patient_id, ward_id, bed_id, predicted_ward, ml_confidence, priority_score)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (patient_id, ward_id, bed_id, predicted_ward, confidence, 5))
        
        
        cursor.execute("UPDATE beds SET is_occupied = 1 WHERE bed_id = %s", (bed_id,))
        cursor.execute("UPDATE wards SET available_beds = available_beds - 1 WHERE ward_id = %s", (ward_id,))
        
        conn.commit()
        
        return {
            'success': True,
            'patient_id': patient_id,
            'predicted_ward': predicted_ward,
            'bed_number': bed_number,
            'ward_id': ward_id,
            'confidence': f"{round(confidence * 100, 2)}%"
        }
        
    except Exception as e:
        conn.rollback()
        return {'success': False, 'message': str(e)}

print("✅ predict_and_assign_ward defined!")

✅ predict_and_assign_ward defined!


In [4]:
def add_patient_and_assign(patient_features, patient_id, patient_name, age, gender):

    cursor = conn.cursor()
    
    
    try:
        cursor.execute("""
            INSERT INTO patients (patient_id, name, age, gender, admission_date, status)
            VALUES (%s, %s, %s, %s, CURDATE(), 'admitted')
        """, (patient_id, patient_name, age, gender))
        conn.commit()
        print(f"✅ Patient {patient_id} added to database")
    except Exception as e:
        print(f"⚠️ Patient insert: {e}")
    
    
    return predict_and_assign_ward(patient_features, patient_id)

print("✅ add_patient_and_assign defined!")

✅ add_patient_and_assign defined!


In [5]:
sample_patient = [
    45, 1, 3, 150, 280, 165, 0, 110, 25, 95, 26, 0.467, 1, 0, 0, 0
]

print(f"✅ Sample patient created with {len(sample_patient)} features")

✅ Sample patient created with 16 features


In [6]:
print("🧪 Testing Complete Workflow...\n")

result = add_patient_and_assign(
    patient_features=sample_patient,
    patient_id='PAT-TEST-005',
    patient_name='Jane Doe',
    age=45,
    gender=0
)

print("📊 Result:")
for key, value in result.items():
    print(f"  {key}: {value}")

if result['success']:
    verify_cursor = conn.cursor()
    
    verify_cursor.execute("""
        SELECT p.patient_id, p.name, p.age, 
               pa.predicted_ward, pa.ml_confidence,
               b.bed_number, pa.assigned_date
        FROM patients p
        JOIN patient_assignments pa ON p.patient_id = pa.patient_id
        JOIN beds b ON pa.bed_id = b.bed_id
        WHERE p.patient_id = %s
    """, (result['patient_id'],))
    
    data = verify_cursor.fetchone()
    print("\n✅ Complete Patient Record:")
    print(f"  Patient ID: {data[0]}")
    print(f"  Name: {data[1]}")
    print(f"  Age: {data[2]}")
    print(f"  Assigned Ward: {data[3]}")
    print(f"  Bed Number: {data[5]}")
    print(f"  ML Confidence: {data[4]*100:.2f}%")
    print(f"  Assigned Date: {data[6]}")
    
    verify_cursor.close()

🧪 Testing Complete Workflow...

⚠️ Patient insert: 1062 (23000): Duplicate entry 'PAT-TEST-005' for key 'patients.PRIMARY'
📊 Result:
  success: True
  patient_id: PAT-TEST-005
  predicted_ward: Cardiology Ward
  bed_number: CAR-007
  ward_id: 3
  confidence: 75.29%

✅ Complete Patient Record:
  Patient ID: PAT-TEST-005
  Name: Jane Doe
  Age: 45
  Assigned Ward: Cardiology Ward
  Bed Number: CAR-003
  ML Confidence: 75.29%
  Assigned Date: 2026-03-04 14:49:31


InternalError: Unread result found